# XLAV — E5: CLIPSeg + SAM (tinh chỉnh mask không cần train)

Nâng cấp **pipeline**: mask của CLIPSeg (decoder E1 epoch 7) được dùng làm prompt cho **SAM** (ViT-B) để vẽ lại biên.
Không train lại gì. Prompt chỉ lấy từ dự đoán của CLIPSeg — ground truth chỉ dùng để chấm.

- Prompt: các vùng liên thông của `p ≥ THRESHOLD` (tối đa `SAM_MAX_COMPONENTS` vùng, bỏ vùng nhỏ hơn
  `SAM_MIN_COMPONENT_RATIO` × vùng lớn nhất); mỗi vùng một box, tuỳ biến thể thêm điểm có xác suất cao nhất.
  Phrase nhiều instance → nhiều box, gộp bằng max logit.
- Biến thể: `clipseg` (gốc), `sam_box_replace`, `sam_box_mean`, `sam_boxpoint_replace`, `sam_boxpoint_mean`.
  `replace` = xác suất SAM; `mean` = trung bình xác suất CLIPSeg và SAM. Query không có vùng nào ≥ ngưỡng giữ nguyên CLIPSeg.
- Chấm theo protocol paper (352×352, 49 ngưỡng, FP32) cho mọi biến thể trên **cùng query**.
- **Chọn trên val, báo trên test:** biến thể SAM có mIoU@0,3 trên val cao nhất được chốt trước khi chấm test.
  Dòng `clipseg` trên test phải trùng số test đã biết của decoder (`CLIPSEG_REFERENCE_TEST`; E1: 46,66 / 54,81 / 77,95)
  — kiểm tra pipeline không lệch.
- Decoder chọn bằng `DECODER_LABEL` + `DECODER_SHA256`: E1 epoch 7 (mặc định), B1, hoặc E4c (checkpoint có LoRA trên
  CLIP vision — notebook tự gắn LoRA, embedding điều kiện vẫn dùng CLIP gốc như lúc train).

**Chạy trên Kaggle:** Add Input `poudelsandeep/phrasecut` và dataset chứa checkpoint decoder (tìm theo SHA256);
GPU T4 (x1 là đủ), Internet on; **Save & Run All**.
Một phiên chấm hết val rồi test (ước tính 3–4 giờ). Kết quả ở `/kaggle/working/clipseg_e5_sam.zip`.


## 1. Dependency

Giữ PyTorch/CUDA, NumPy, Pillow, scikit-learn và matplotlib sẵn có của Kaggle.
Cell này chỉ pin Transformers và gdown; không cài wheel PyTorch CPU.
Nếu kernel đã import Transformers phiên bản khác trước đó, restart session rồi chạy lại từ đầu.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.55.4", "gdown==5.2.0",
])


## 2. Cấu hình

Mặc định tải dữ liệu online. Nếu đã Add Input một Kaggle Dataset chứa dữ liệu,
đặt `DATA_ROOT` tới thư mục chứa `refer_<EVAL_SPLIT>.json`, `image_data_split.json`
(hoặc `image_data_split3000.json`) và `images/`. Để `None` sẽ tự tìm trong `/kaggle/input`
rồi tải từ nguồn chính thức nếu chưa có. `MODEL_PATH=None` tải Hugging Face;
muốn offline thì trỏ tới thư mục snapshot có config, processor, tokenizer và weights.

Baseline mặc định FP32, dùng một GPU; T4 x2 vẫn chạy trên GPU đầu tiên.
`USE_AMP=True` là tùy chọn FP16, phải ghi riêng khi so sánh số liệu.


In [ ]:
from pathlib import Path

SAM_MODEL = "facebook/sam-vit-base"   # SAM ViT-B trong Transformers 4.55.4
SAM_MAX_COMPONENTS = 4                # số vùng liên thông tối đa làm prompt cho mỗi query
SAM_MIN_COMPONENT_RATIO = 0.1         # bỏ vùng nhỏ hơn tỉ lệ này × vùng lớn nhất
DECODER_SHA256 = "b9bc28509a6a6f36495bc4826a511d182bdee8885a819190997da56fd67947df"  # E1 epoch 7; DECODER_CHECKPOINT=None thì tự tìm trong Kaggle Input
DECODER_LABEL = "e1"                  # "e1" | "b1" | "e4c_lr003": tên decoder ghi vào kết quả
CLIPSEG_REFERENCE_TEST = {"miou_0.3": 0.4666, "fgiou_0.3": 0.5481, "ap": 0.7795}  # số test đã biết của decoder này
LORA_RANK = 8                         # chỉ dùng khi checkpoint có LoRA nhưng không ghi cấu hình
LORA_ALPHA = 16
LORA_GRAD_CHECKPOINT = False
VARIANTS = ["clipseg", "sam_box_replace", "sam_box_mean", "sam_boxpoint_replace", "sam_boxpoint_mean"]

DATA_ROOT = None                 # ví dụ: "/kaggle/input/phrasecut-miniv/VGPhraseCut_v0"
MODEL_PATH = None                # thư mục checkpoint offline, hoặc None
IMAGE_ROOT = None                # thư mục chứa ảnh <image_id>.jpg; None = tìm trong mọi Kaggle Input
WORK_ROOT = Path("/kaggle/working")
REQUIRE_GPU = True               # False chỉ khi kiểm tra trên máy CPU
EVAL_SPLIT = "val"             # "miniv" (100 ảnh) | "val" (~2.900 ảnh) | "test" (2.545 ảnh, split của paper)
MAX_QUERIES = None               # None = toàn bộ split; 8 = smoke
BATCH_SIZE = 8
EPOCHS = 1
TRAIN_DECODER = False            # E1 = False: chỉ đánh giá checkpoint pretrained
TRAIN_ROOT = None                # thư mục refer_train.json + images/ trong Kaggle Input
MAX_TRAIN_QUERIES = None         # None = toàn bộ train; 8 = smoke train
TRAIN_INIT = "random_decoder"   # "pretrained_decoder" = fine-tune decoder đã học
TRAIN_MODE = "phrasecut_plus"    # "phrasecut_plus" = text + ảnh support như paper; "text_only" = ablation rd64-uni-novis
NEGATIVE_PROB = 0.2              # tỉ lệ mẫu âm: cụm từ không có trong ảnh, mask rỗng
MIX_TEXT_MAX = 0.5               # trọng số text w ~ U(0, 0,5); mẫu không có support dùng w = 1
PROMPT_AUGMENT = True            # train với 9 template "shuffle+" của tác giả
AUG_CROP = True                  # crop vuông ngẫu nhiên, giữ vật trong khung
LEARNING_RATE = 1e-3
MIN_LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
NUM_WORKERS = 4                  # worker tạo mẫu PhraseCut+ (Kaggle có 4 CPU); 0 khi chạy trên Windows
MULTI_GPU = True                 # train chia mỗi batch cho mọi GPU (T4 x2) bằng DataParallel
RESUME_CHECKPOINT = None         # checkpoint epoch của chính notebook, hoặc None
USE_AMP = False                  # FP16 autocast khi TRAIN (nên True trên T4)
EVAL_AMP = False                 # đánh giá FP32 như score.py của tác giả
DECODER_CHECKPOINT = None        # TRAIN_DECODER=False: đánh giá decoder_epoch_XXX.pt đã train thay cho decoder tác giả
SEED = 42
THRESHOLD = 0.3
CALIBRATION_FRACTION = 0.2
THRESHOLD_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
PROMPT_TEMPLATE = "{phrase}"      # giữ nguyên phrase, cố định trước evaluation
SAVE_PROBABILITIES = False       # True lưu float32 .npy (tốn thêm dung lượng)
N_VISUALIZATIONS = 6
DOWNLOAD_WORKERS = 4

MODEL_ID = "CIDAS/clipseg-rd64-refined"
MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"
PHRASECUT_API_COMMIT = "276470b7773d8a230aee5b5536876fbc485fb04b"


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import shutil
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext

import gdown
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor
from IPython.display import display

assert BATCH_SIZE >= 1
assert isinstance(EPOCHS, int) and EPOCHS >= 1
assert TRAIN_INIT in ["random_decoder", "pretrained_decoder"]
assert TRAIN_MODE in ["phrasecut_plus", "text_only"]
assert EVAL_SPLIT in ["miniv", "val", "test"]
assert not (TRAIN_DECODER and DECODER_CHECKPOINT), "DECODER_CHECKPOINT chỉ dùng khi TRAIN_DECODER=False."
assert 0 <= NEGATIVE_PROB < 1 and 0 <= MIX_TEXT_MAX <= 1
assert MAX_TRAIN_QUERIES is None or MAX_TRAIN_QUERIES >= 1
assert 0 <= MIN_LEARNING_RATE <= LEARNING_RATE and LEARNING_RATE > 0
assert MAX_QUERIES is None or MAX_QUERIES >= 1
assert 0 < CALIBRATION_FRACTION < 1
assert 0 <= THRESHOLD <= 1 and all(0 <= t <= 1 for t in THRESHOLD_GRID)
assert "{phrase}" in PROMPT_TEMPLATE
random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA. Bật GPU trong Kaggle Settings rồi restart session.")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        for index in range(torch.cuda.device_count()):
            torch.ones(1, device=f"cuda:{index}").add_(1).item()
    except RuntimeError as exc:
        raise RuntimeError("PyTorch của Kaggle không chạy được trên GPU này (thường gặp với P100). "
                           "Chọn Accelerator GPU T4 x2 rồi restart session.") from exc
assert not (USE_AMP or EVAL_AMP) or device.type == "cuda", "FP16 AMP yêu cầu CUDA."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUT = WORK_ROOT / "clipseg_e5_sam"
OUT.mkdir(parents=True, exist_ok=True)
# Xóa dấu hoàn thành cũ trước khi bắt đầu lượt mới.
(OUT / "summary.json").unlink(missing_ok=True)
(OUT / "run.json").unlink(missing_ok=True)
(WORK_ROOT / "clipseg_e5_sam.zip").unlink(missing_ok=True)

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

versions = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "numpy", "Pillow", "scikit-learn", "matplotlib", "gdown"]}
print("Device:", device, "| GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
      if device.type == "cuda" else "CPU")
print(json.dumps(versions, indent=2))


In [ ]:
# Checkpoint decoder trong Kaggle Input, tìm theo SHA256 (tên file có thể là decoder_epoch_002/007.pt).
if DECODER_CHECKPOINT is None:
    found = sorted(Path("/kaggle/input").rglob("*.pt")) if Path("/kaggle/input").exists() else []
    found = [p for p in found if p.stat().st_size < 500 * 1024**2 and sha256(p) == DECODER_SHA256]
    if not found:
        raise FileNotFoundError(f"Không tìm thấy checkpoint {DECODER_LABEL} (SHA256 {DECODER_SHA256[:8]}…) trong Kaggle Input.")
    DECODER_CHECKPOINT = str(found[0])
assert sha256(Path(DECODER_CHECKPOINT)) == DECODER_SHA256, f"Checkpoint khác {DECODER_LABEL}."
print("Decoder:", DECODER_LABEL, DECODER_CHECKPOINT)


## 3. Lấy annotation chính thức và chọn query

Chỉ tải `refer_<EVAL_SPLIT>.json` và metadata ảnh từ Drive ID trong
[downloader tác giả](https://github.com/ChenyunWu/PhraseCutDataset/blob/276470b7773d8a230aee5b5536876fbc485fb04b/download_dataset.py).
Chỉ ảnh của các query được chọn mới được tải, không tải toàn bộ Visual Genome. Với `test`/`val`
(khoảng 2.500–2.900 ảnh) nên Add Input dataset đã có ảnh để khỏi tải lại mỗi lượt.
Nếu Google Drive hết quota, Add Input dữ liệu đã tải rồi đặt `DATA_ROOT`.

Chia calibration/evaluation theo **image_id**, seed 42, trước khi đọc prediction.
Nếu giới hạn số query, sampling bằng seed, lưu đúng danh sách vào `selected_tasks.json`.


In [ ]:
# Drive ID của annotation chính thức, theo download_dataset.py của PhraseCut.
REFER_DRIVE_IDS = {"miniv": "1oLcTQ1blTIQyu5ZMelQN2HSniuQaxM4E", "val": "1UyojArOFPlsSeNbA9fHWjCjOOU-OCohG",
                   "test": "1jrzXm1gcq6f5hNDeamZd0UmyyHUv61IZ", "train": "1qx-0q6r9r0YUGpoyT0B8HJKmUFWQDSu7"}

def metadata_path(root):
    for name in ["image_data_split.json", "image_data_split3000.json"]:
        candidate = root / name
        if candidate.is_file():
            return candidate
    return root / "image_data_split.json"

if DATA_ROOT is None:
    candidates = sorted(Path("/kaggle/input").rglob(f"refer_{EVAL_SPLIT}.json")) if Path("/kaggle/input").exists() else []
    candidates = [p.parent for p in candidates if metadata_path(p.parent).is_file()]
    if len(candidates) > 1:
        raise ValueError(f"Có nhiều dataset chứa refer_{EVAL_SPLIT}.json; đặt DATA_ROOT rõ ràng: " + str(candidates))
    data_root = candidates[0] if candidates else WORK_ROOT / "phrasecut_annotations"
else:
    data_root = Path(DATA_ROOT)
    if not data_root.is_dir():
        raise FileNotFoundError(f"DATA_ROOT không tồn tại: {data_root}")
data_root.mkdir(parents=True, exist_ok=True)
annotation_path = data_root / f"refer_{EVAL_SPLIT}.json"
info_path = metadata_path(data_root)

def ensure_json(path, file_id):
    if path.is_file():
        value = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError(f"JSON không phải danh sách annotation: {path}")
        return value
    if str(path).startswith("/kaggle/input/"):
        raise FileNotFoundError(f"Thiếu {path.name} trong input read-only; bổ sung Kaggle Dataset.")
    temp = path.with_suffix(".download")
    try:
        result = gdown.download(id=file_id, output=str(temp), quiet=False)
        if result is None:
            raise RuntimeError("Google Drive không trả về file.")
        value = json.loads(temp.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError("Nội dung download không phải annotation JSON.")
        temp.replace(path)
        return value
    except Exception as exc:
        raise RuntimeError(f"Không tải được {path.name}. Bật Internet, hoặc Add Input dataset và đặt DATA_ROOT.") from exc

tasks = ensure_json(annotation_path, REFER_DRIVE_IDS[EVAL_SPLIT])
image_info = ensure_json(info_path, "1xB9eExJo35K3DQu8PnBWZ1q_OMoMl8-i")
info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == EVAL_SPLIT}
assert len({str(t["task_id"]) for t in tasks}) == len(tasks), "Task ID bị trùng."
assert all(int(t["image_id"]) in info_by_id for t in tasks), f"Annotation không thuộc split {EVAL_SPLIT}."
assert all(t.get("Polygons") and t.get("phrase", "").strip() for t in tasks), "Thiếu polygon GT hoặc phrase."
assert all(re.fullmatch(r"[A-Za-z0-9_-]+", str(t["task_id"])) for t in tasks), "Task ID không hợp lệ."

selected = sorted(tasks, key=lambda t: str(t["task_id"]))
if MAX_QUERIES is not None and MAX_QUERIES < len(selected):
    selected = random.Random(SEED).sample(selected, MAX_QUERIES)
selected.sort(key=lambda t: (int(t["image_id"]), str(t["task_id"])))
image_ids = sorted({int(t["image_id"]) for t in selected})
if len(image_ids) < 2:
    raise ValueError("Cần ít nhất 2 image_id để chia calibration/evaluation; tăng MAX_QUERIES.")
shuffled_ids = image_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)
n_cal = min(len(image_ids) - 1, max(1, round(len(image_ids) * CALIBRATION_FRACTION)))
calibration_ids = set(shuffled_ids[:n_cal])
group_for_image = {i: "calibration" if i in calibration_ids else "evaluation" for i in image_ids}
assert not calibration_ids.intersection(set(image_ids) - calibration_ids)
write_json(OUT / "selected_tasks.json", [{"task_id": str(t["task_id"]), "image_id": int(t["image_id"]),
    "phrase": t["phrase"], "group": group_for_image[int(t["image_id"])]} for t in selected])
print(f"Annotation {EVAL_SPLIT}: {len(tasks)} queries / {len(info_by_id)} images")
print(f"Selected: {len(selected)} queries / {len(image_ids)} images; calibration {n_cal} images")


## 4. Tải và kiểm tra ảnh

Ưu tiên ảnh đã có trong dataset input. Ảnh thiếu sẽ tải từ URL Visual Genome trong metadata
bằng HTTPS vào working cache, retry có giới hạn. Lỗi tải hoặc kích thước khác metadata làm
dừng lượt chạy: không âm thầm bỏ query để tạo số liệu đẹp hơn.


In [ ]:
image_cache = WORK_ROOT / "phrasecut_image_cache"
image_cache.mkdir(parents=True, exist_ok=True)
def image_search_roots(*extra):
    """Nơi tìm ảnh <image_id>.jpg: IMAGE_ROOT nếu đặt, nếu không thì mọi Kaggle Input; cộng thư mục annotation."""
    roots = [Path(IMAGE_ROOT)] if IMAGE_ROOT else ([Path("/kaggle/input")] if Path("/kaggle/input").exists() else [])
    roots += [Path(r) for r in extra]
    # Bỏ thư mục nằm trong thư mục đã quét để không quét hai lần.
    return [r for i, r in enumerate(roots) if r.is_dir() and not any(r == q or q in r.parents for q in roots[:i])]

def index_local_images(roots, wanted_ids):
    index = {}
    for search_root in roots:
        for path in sorted(search_root.rglob("*")):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and path.stem.isdigit() and int(path.stem) in wanted_ids:
                index.setdefault(int(path.stem), path)
    return index

local_images = index_local_images(image_search_roots(data_root), set(image_ids))
print(f"Ảnh có sẵn trong Input: {len(local_images)}/{len(image_ids)}; phần còn lại tải từ Visual Genome")

def valid_image(path, info):
    with Image.open(path) as image:
        image.load()
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"{path.name}: ảnh {image.size} != metadata {(info['width'], info['height'])}")

def prepare_image(image_id):
    info = info_by_id[image_id]
    path = local_images.get(image_id, image_cache / f"{image_id}.jpg")
    if path.is_file():
        valid_image(path, info)
        return image_id, path
    url = info["url"].replace("http://", "https://", 1)
    temp = path.with_suffix(".download")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with temp.open("wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        f.write(chunk)
            valid_image(temp, info)
            temp.replace(path)
            return image_id, path
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

image_paths, image_errors = {}, []
def prepare_safe(image_id):
    try:
        return prepare_image(image_id), None
    except Exception as exc:
        return None, {"image_id": image_id, "error": str(exc)}

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    for result, error in tqdm(pool.map(prepare_safe, image_ids), total=len(image_ids), desc="Images"):
        if error:
            image_errors.append(error)
        else:
            image_paths[result[0]] = result[1]
write_json(OUT / "image_errors.json", image_errors)
if image_errors:
    raise RuntimeError(f"{len(image_errors)} ảnh lỗi; xem image_errors.json. Sửa dữ liệu rồi chạy lại.")
print("Validated images:", len(image_paths))


## 5. Nạp model và định nghĩa inference/metric

Dùng [triển khai CLIPSeg của Transformers](https://huggingface.co/docs/transformers/v4.55.4/en/model_doc/clipseg),
revision checkpoint được cố định. Tiền xử lý, rasterize GT và metric bám theo code đánh giá của tác giả
([score.py](https://github.com/timojl/clipseg/blob/master/score.py),
[metrics.py](https://github.com/timojl/clipseg/blob/master/metrics.py) `FixedIntervalMetrics`,
[datasets/phrasecut.py](https://github.com/timojl/clipseg/blob/master/datasets/phrasecut.py)):

- Ảnh resize **352×352** bằng bilinear `align_corners=True`, chia 255, chuẩn hóa mean/std; text giữ nguyên phrase.
- GT: hợp polygon bằng `skimage.draw.polygon2mask`, rồi resize nearest về 352×352. Chấm **ở 352×352**, sigmoid, FP32.
- 49 ngưỡng 0,02–0,98, pixel dương khi `p > t`. Với từng ngưỡng: `miou` = trung bình theo query của
  `tp/(1+tp+fp+fn)`; `fgiou` = cộng dồn mọi pixel; AP = tích phân Simpson của đường precision–recall cộng dồn.
- Bảng paper dùng `miou_0.3`, `fgiou_0.3`, `ap` (cột `pc_miou_0.3`, `pc_fgiou_0.3`, `pc_ap` trong `experiments/phrasecut.yaml`).

Bên cạnh đó vẫn giữ metric ở độ phân giải ảnh gốc (IoU theo query, cumulative IoU, AP pixel theo query)
để so với các lượt chạy trước. Không dùng GT để tạo text prompt hoặc thay đổi đầu vào ảnh.


In [ ]:
load_kwargs = {"local_files_only": True} if MODEL_PATH else {"revision": MODEL_REVISION}
model_source = str(MODEL_PATH) if MODEL_PATH else MODEL_ID
processor = CLIPSegProcessor.from_pretrained(model_source, use_fast=False, **load_kwargs)
model = CLIPSegForImageSegmentation.from_pretrained(model_source, **load_kwargs).to(device).eval()
assert processor.image_processor.size == {"height": 352, "width": 352}, "Processor không có input 352 như baseline."
assert model.config.vision_config.patch_size == 16, "Baseline yêu cầu ViT-B/16."
print("Model:", model_source, "| parameters:", sum(p.numel() for p in model.parameters()))

try:
    from skimage.draw import polygon2mask
except ImportError as exc:
    raise ImportError("Cần scikit-image (có sẵn trên Kaggle); cài bằng pip install scikit-image.") from exc
try:
    from scipy.integrate import simpson
except ImportError:  # SciPy cũ
    from scipy.integrate import simps as simpson

# Tiền xử lý và độ phân giải chấm điểm theo score.py + datasets/phrasecut.py của CLIPSeg.
PAPER_SIZE = 352
PAPER_THRESHOLDS = np.linspace(0, 1, 51)[1:-1]   # 49 ngưỡng 0,02–0,98 như FixedIntervalMetrics
image_mean = torch.tensor(processor.image_processor.image_mean, dtype=torch.float32).view(3, 1, 1)
image_std = torch.tensor(processor.image_processor.image_std, dtype=torch.float32).view(3, 1, 1)

def rasterize_gt(task, width, height, allow_empty=False):
    """Hợp polygon mọi instance bằng skimage polygon2mask (x, y đảo thành hàng, cột), như PhraseCut.load_sample."""
    target = np.zeros((height, width), dtype=bool)
    for instance in task["Polygons"]:
        for polygon in instance:
            if len(polygon) >= 3:
                target |= polygon2mask((height, width), np.asarray(polygon, dtype=np.float64)[:, ::-1])
    if not allow_empty and not target.any():
        raise ValueError(f"GT rỗng: task {task['task_id']}")
    return target

def paper_pixel_values(image):
    """Ảnh RGB → [3,352,352]: resize bilinear align_corners=True trên ảnh gốc, /255, chuẩn hóa mean/std."""
    pixels = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1).unsqueeze(0)
    pixels = F.interpolate(pixels, size=(PAPER_SIZE, PAPER_SIZE), mode="bilinear", align_corners=True)[0]
    return (pixels / 255 - image_mean) / image_std

def paper_target(target):
    """GT ở độ phân giải gốc → 352×352 bằng nearest, như nnf.interpolate trong load_sample."""
    mask = torch.from_numpy(target.astype(np.uint8))[None, None]
    return F.interpolate(mask, size=(PAPER_SIZE, PAPER_SIZE), mode="nearest")[0, 0].numpy().astype(bool)

def paper_threshold_stats(probability, target):
    """tp, fp, fn, tn tại 49 ngưỡng (dương khi p > t) cho một query, như FixedIntervalMetrics.add."""
    order = np.argsort(probability, axis=None, kind="stable")
    p, g = probability.ravel()[order], target.ravel()[order]
    below = np.searchsorted(p, PAPER_THRESHOLDS, side="right")   # số pixel p <= t
    fn = np.concatenate([[0], np.cumsum(g, dtype=np.int64)])[below]
    tp = int(g.sum()) - fn
    fp = (len(p) - below) - tp
    tn = below - fn
    return np.stack([tp, fp, fn, tn]).astype(np.int64)

def paper_metrics(stats):
    """Metric bảng PhraseCut của paper từ stats [Q,4,49]: miou (trung bình theo query), fgiou (cộng dồn), AP Simpson."""
    stats = np.asarray(stats, dtype=np.float64)
    tp, fp, fn = stats[:, 0], stats[:, 1], stats[:, 2]
    s_tp, s_fp, s_fn = tp.sum(0), fp.sum(0), fn.sum(0)
    made = (s_tp + s_fp) > 0
    pairs = sorted(set(zip((s_tp / (1 + s_tp + s_fn))[made], (s_tp / (1 + s_tp + s_fp))[made])))
    recalls, precisions = (np.array(v) for v in zip(*pairs))
    fgiou = s_tp / (1 + s_tp + s_fp + s_fn)
    miou = (tp / (1 + tp + fp + fn)).mean(0)
    index = {t: int(np.argmin(np.abs(PAPER_THRESHOLDS - t))) for t in (0.3, 0.5)}
    return {"n_queries": len(stats), "miou_0.3": float(miou[index[0.3]]), "fgiou_0.3": float(fgiou[index[0.3]]),
            "fgiou_0.5": float(fgiou[index[0.5]]), "ap": float(simpson(precisions, x=recalls)),
            "miou_best": float(miou.max()), "miou_best_t": float(PAPER_THRESHOLDS[miou.argmax()]),
            "fgiou_best": float(fgiou.max()), "fgiou_best_t": float(PAPER_THRESHOLDS[fgiou.argmax()])}

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)

@torch.inference_mode()
def predict_batch(image, batch_tasks, with_paper=False):
    """Xác suất ở kích thước ảnh gốc; with_paper=True trả thêm xác suất 352×352 để chấm theo paper."""
    prompts = [PROMPT_TEMPLATE.format(phrase=t["phrase"]) for t in batch_tasks]
    inputs = dict(processor.tokenizer(text=prompts, padding=True, truncation=True, return_tensors="pt"))
    inputs["pixel_values"] = paper_pixel_values(image).unsqueeze(0).repeat(len(prompts), 1, 1, 1)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    synchronize()
    started = time.perf_counter()
    context = torch.autocast(device_type="cuda", dtype=torch.float16) if EVAL_AMP else nullcontext()
    with context:
        logits = model(**inputs).logits
    logits = logits.float().reshape(len(prompts), *logits.shape[-2:])
    probabilities = F.interpolate(logits.unsqueeze(1), size=(image.height, image.width),
        mode="bilinear", align_corners=False).squeeze(1).sigmoid()
    synchronize()
    seconds = time.perf_counter() - started
    values = probabilities.cpu().numpy()
    if not np.isfinite(values).all():
        raise ValueError("Prediction chứa NaN/Inf; thử EVAL_AMP=False.")
    if with_paper:
        return values, logits.sigmoid().cpu().numpy(), seconds
    return values, seconds

def binary_metrics(probability, target, threshold):
    if probability.shape != target.shape or not target.any():
        raise ValueError("Shape không khớp hoặc GT rỗng.")
    prediction = probability >= threshold
    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    return {"intersection": intersection, "union": union, "iou": intersection / union,
            "dice": 2 * intersection / int(prediction.sum() + target.sum())}

def summarize(rows):
    if not rows:
        raise ValueError("Không có query để tính metric.")
    result = {"n_queries": len(rows), "n_images": len({r["image_id"] for r in rows}),
        "mean_sample_iou": float(np.mean([r["iou"] for r in rows])),
        "cumulative_iou": sum(r["intersection"] for r in rows) / sum(r["union"] for r in rows),
        "mean_dice": float(np.mean([r["dice"] for r in rows]))}
    for cutoff in [0.3, 0.5, 0.7, 0.9]:
        result[f"precision_at_{cutoff}"] = float(np.mean([r["iou"] >= cutoff for r in rows]))
    if "pixel_ap" in rows[0]:
        result["mean_sample_pixel_ap"] = float(np.mean([r["pixel_ap"] for r in rows]))
    return result


In [ ]:
from torch.utils.data import Dataset, DataLoader

# 9 template "shuffle+" trong models/clipseg.py của repo CLIPSeg gốc.
SHUFFLE_PLUS_TEMPLATES = ["a photo of a {}.", "a photograph of a {}.", "an image of a {}.", "{}.",
    "a cropped photo of a {}.", "a good photo of a {}.", "a photo of one {}.", "a bad photo of a {}.",
    "a photo of the {}."]

def random_square_crop(target, iterations=50, min_frac=0.05):
    """Crop vuông cạnh min(H, W); nhận crop đầu tiên có số pixel vật > min_frac × diện tích ảnh (find_crop gốc)."""
    height, width = target.shape
    side = min(height, width)
    min_sum = height * width * min_frac
    best = (-1, 0, 0)
    for _ in range(iterations):
        y = int(torch.randint(0, height - side + 1, (1,)))
        x = int(torch.randint(0, width - side + 1, (1,)))
        total = int(target[y:y + side, x:x + side].sum())
        if total > min_sum:
            return y, x, side
        if total > best[0]:
            best = (total, y, x)
    return best[1], best[2], side

def gaussian_blur(image, kernel_size=15, sigma=3.0):
    """Gaussian blur [C,H,W] biên reflect, tương đương cv2.GaussianBlur(img, (15, 15), 3) của repo gốc."""
    coords = torch.arange(kernel_size, dtype=image.dtype) - (kernel_size - 1) / 2
    kernel = torch.exp(-coords ** 2 / (2 * sigma ** 2))
    kernel = kernel / kernel.sum()
    channels, pad = image.shape[0], kernel_size // 2
    blurred = F.pad(image.unsqueeze(0), (pad, pad, pad, pad), mode="reflect")
    blurred = F.conv2d(blurred, kernel.view(1, 1, 1, -1).repeat(channels, 1, 1, 1), groups=channels)
    blurred = F.conv2d(blurred, kernel.view(1, 1, -1, 1).repeat(channels, 1, 1, 1), groups=channels)
    return blurred.squeeze(0)

def visual_prompt(image, mask, size=352, blur_sigma=3.0, background_factor=0.1, context=0.1):
    """crop_blur_highlight352: nền làm mờ còn 10% (trên ảnh đã chuẩn hóa), crop quanh vật nới 10%, pad vuông, resize."""
    mask = mask.to(image.dtype)
    blended = image * mask + background_factor * gaussian_blur(image, sigma=blur_sigma) * (1 - mask)
    ys, xs = torch.nonzero(mask > 0, as_tuple=True)
    if len(ys):
        y0, y1, x0, x1 = int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1
    else:
        y0, y1, x0, x1 = 0, mask.shape[0], 0, mask.shape[1]
    dy, dx = context * (y1 - y0), context * (x1 - x0)
    y0, y1 = int(max(0, y0 - dy)), int(min(mask.shape[0], y1 + dy))
    x0, x1 = int(max(0, x0 - dx)), int(min(mask.shape[1], x1 + dx))
    crop = blended[:, y0:y1, x0:x1]
    height, width = crop.shape[1:]
    side = max(height, width)
    top, left = (side - height) // 2, (side - width) // 2
    crop = F.pad(crop, (left, side - width - left, top, side - height - top))
    return F.interpolate(crop.unsqueeze(0), size=(size, size), mode="nearest").squeeze(0)

def sample_text_weights(support_valid, max_text_weight):
    """w ~ U(0, max) theo từng mẫu; mẫu không có support dùng w = 1. Lấy mẫu trên CPU trước khi chia GPU."""
    weights = torch.rand(len(support_valid), 1) * max_text_weight
    return torch.maximum(weights, (~support_valid.cpu()).float()[:, None])

def mix_conditionals(text_cond, visual_cond, text_weights):
    """cond = w·text + (1−w)·ảnh."""
    weights = text_weights.to(text_cond.device, text_cond.dtype)
    return text_cond * weights + visual_cond * (1 - weights)

training_metadata = {"enabled": TRAIN_DECODER}
if TRAIN_DECODER:
    if TRAIN_ROOT is None:
        train_candidates = sorted(Path("/kaggle/input").rglob("refer_train.json")) if Path("/kaggle/input").exists() else []
        if len(train_candidates) > 1:
            raise ValueError("Có nhiều dataset train; đặt TRAIN_ROOT rõ ràng.")
        train_root = train_candidates[0].parent if train_candidates else WORK_ROOT / "phrasecut_train"
    else:
        train_root = Path(TRAIN_ROOT)
        if not train_root.is_dir():
            raise FileNotFoundError(f"TRAIN_ROOT không tồn tại: {train_root}")
    train_root.mkdir(parents=True, exist_ok=True)
    train_annotation_path = train_root / "refer_train.json"
    train_raw = ensure_json(train_annotation_path, REFER_DRIVE_IDS["train"])
    train_info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == "train"}
    # Các image_id sai kích thước được loại trong wrapper CLIPSeg gốc.
    bad_image_ids = {150417, 285665, 498246, 61564, 285743, 498269, 498010, 150516,
                     150344, 286093, 61530, 150333, 286065, 285814, 498187, 285761, 498042}
    train_tasks = []
    excluded_train = []
    for task in train_raw:
        image_id = int(task["image_id"])
        if image_id not in train_info_by_id:
            raise ValueError(f"Task {task['task_id']} không thuộc split train chính thức.")
        if image_id in bad_image_ids:
            excluded_train.append({"task_id": str(task["task_id"]), "image_id": image_id,
                                   "reason": "known_dimension_mismatch_in_original_clipseg_wrapper"})
            continue
        if not task.get("Polygons") or not task.get("phrase", "").strip():
            raise ValueError(f"Train task thiếu GT hoặc phrase: {task['task_id']}")
        train_tasks.append({key: task[key] for key in ["task_id", "image_id", "phrase", "Polygons"]})
    del train_raw
    train_tasks.sort(key=lambda t: str(t["task_id"]))
    full_train_queries = len(train_tasks)
    if MAX_TRAIN_QUERIES is not None and MAX_TRAIN_QUERIES < len(train_tasks):
        train_tasks = random.Random(SEED).sample(train_tasks, MAX_TRAIN_QUERIES)
    if not train_tasks:
        raise ValueError("Train dataset rỗng.")
    train_image_ids = {int(t["image_id"]) for t in train_tasks}
    assert train_image_ids.isdisjoint(info_by_id), f"Train/{EVAL_SPLIT} bị trùng image_id."
    assert len({str(t["task_id"]) for t in train_tasks}) == len(train_tasks), "Train task ID bị trùng."
    write_json(OUT / "excluded_train_tasks.json", excluded_train)
    write_json(OUT / "train_tasks.json", [{"task_id": str(t["task_id"]), "image_id": int(t["image_id"]),
                                          "phrase": t["phrase"]} for t in train_tasks])
    train_local_images = index_local_images(image_search_roots(train_root), train_image_ids)
    print(f"Ảnh train có sẵn trong Input: {len(train_local_images)}/{len(train_image_ids)}; "
          "phần còn lại tải từ Visual Genome trong lúc train")
    train_image_cache = WORK_ROOT / "phrasecut_train_image_cache"
    train_image_cache.mkdir(parents=True, exist_ok=True)

    def get_train_image_path(image_id):
        path = train_local_images.get(image_id, train_image_cache / f"{image_id}.jpg")
        info = train_info_by_id[image_id]
        if path.is_file():
            return path
        # Temp riêng cho từng worker, tránh ghi đè nếu nhiều query có cùng ảnh.
        temp = path.with_name(f"{image_id}.{os.getpid()}.download")
        url = info["url"].replace("http://", "https://", 1)
        for attempt in range(3):
            try:
                with requests.get(url, stream=True, timeout=(10, 60)) as response:
                    response.raise_for_status()
                    with temp.open("wb") as f:
                        for chunk in response.iter_content(1024 * 1024):
                            f.write(chunk)
                valid_image(temp, info)
                temp.replace(path)
                return path
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(attempt + 1)

    # Chỉ mục cụm từ -> các mẫu train, dùng để chọn support và mẫu âm.
    tasks_by_phrase = defaultdict(list)
    for index, task in enumerate(train_tasks):
        tasks_by_phrase[task["phrase"]].append(index)
    all_train_phrases = sorted(tasks_by_phrase)
    queries_with_support = sum(1 for task in train_tasks if any(
        train_tasks[i]["image_id"] != task["image_id"] for i in tasks_by_phrase[task["phrase"]]))

    def load_train_sample(task):
        """Ảnh chuẩn hóa [3,352,352] và mask [352,352], crop vuông ngẫu nhiên như load_sample gốc."""
        image_id = int(task["image_id"])
        with Image.open(get_train_image_path(image_id)) as raw:
            image = raw.convert("RGB")
        info = train_info_by_id[image_id]
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"Train ảnh {image_id} sai kích thước metadata.")
        # Như bản gốc: polygon quá mảnh có thể cho GT rỗng; mẫu đó vẫn được train với mask rỗng.
        target = rasterize_gt(task, image.width, image.height, allow_empty=True)
        pixels = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1)
        if AUG_CROP:
            y, x, side = random_square_crop(target)
            target, pixels = target[y:y + side, x:x + side], pixels[:, y:y + side, x:x + side]
        pixels = F.interpolate(pixels.unsqueeze(0), size=(352, 352), mode="bilinear", align_corners=True).squeeze(0)
        mask = F.interpolate(torch.from_numpy(target.astype(np.float32))[None, None], size=(352, 352), mode="nearest")[0, 0]
        return (pixels / 255 - image_mean) / image_std, mask

    class PhraseCutPlusDataset(Dataset):
        """PhraseCut+ tạo lại ở mỗi lần lấy mẫu: mẫu âm, support cùng cụm từ từ ảnh khác, template ngẫu nhiên."""
        def __len__(self):
            return len(train_tasks)

        def __getitem__(self, index):
            task = train_tasks[index]
            image, target = load_train_sample(task)
            phrase = task["phrase"]
            if len(all_train_phrases) > 1 and float(torch.rand(1)) < NEGATIVE_PROB:
                while phrase == task["phrase"]:
                    phrase = all_train_phrases[int(torch.randint(0, len(all_train_phrases), (1,)))]
                target = torch.zeros_like(target)
            support, support_valid = torch.zeros_like(image), False
            if TRAIN_MODE == "phrasecut_plus":
                candidates = [i for i in tasks_by_phrase[phrase] if train_tasks[i]["image_id"] != task["image_id"]]
                if candidates:
                    other = train_tasks[candidates[int(torch.randint(0, len(candidates), (1,)))]]
                    support, support_valid = visual_prompt(*load_train_sample(other)), True
            if PROMPT_AUGMENT:
                prompt = SHUFFLE_PLUS_TEMPLATES[int(torch.randint(0, len(SHUFFLE_PLUS_TEMPLATES), (1,)))].format(phrase)
            else:
                prompt = PROMPT_TEMPLATE.format(phrase=phrase)
            return image, prompt, support, support_valid, target

    def collate_train(samples):
        images, prompts, supports, valid, masks = zip(*samples)
        inputs = dict(processor.tokenizer(text=list(prompts), padding=True, truncation=True, return_tensors="pt"))
        inputs.update(pixel_values=torch.stack(images), support_pixel_values=torch.stack(supports),
                      support_valid=torch.tensor(valid), labels=torch.stack(masks))
        return inputs

    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(PhraseCutPlusDataset(), batch_size=BATCH_SIZE, shuffle=True,
        drop_last=False, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
        collate_fn=collate_train, generator=generator)
    steps_per_epoch = len(train_loader)
    total_steps = EPOCHS * steps_per_epoch
    assert steps_per_epoch == math.ceil(len(train_tasks) / BATCH_SIZE)
    training_metadata = {"enabled": True, "split": "train", "initialization": TRAIN_INIT,
        "seed": SEED, "precision": "fp16_autocast" if USE_AMP else "fp32", "prompt_template": PROMPT_TEMPLATE,
        "base_model_source": model_source, "base_model_revision": MODEL_REVISION if not MODEL_PATH else None,
        "full_valid_train_queries": full_train_queries, "selected_train_queries": len(train_tasks),
        "train_images": len(train_image_ids), "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "steps_per_epoch": steps_per_epoch, "total_planned_steps": total_steps,
        "drop_last": False, "lr": LEARNING_RATE, "min_lr": MIN_LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "num_workers": NUM_WORKERS,
        "train_annotation_sha256": sha256(train_annotation_path),
        "train_manifest_sha256": sha256(OUT / "train_tasks.json"),
        "excluded_train_queries": len(excluded_train), "loss": "BCEWithLogitsLoss",
        "train_mode": TRAIN_MODE, "negative_prob": NEGATIVE_PROB,
        "mix_text_weight": [0.0, MIX_TEXT_MAX] if TRAIN_MODE == "phrasecut_plus" else None,
        "prompt_augment": "shuffle+" if PROMPT_AUGMENT else None, "aug_crop": AUG_CROP,
        "visual_prompt": "crop_blur_highlight352" if TRAIN_MODE == "phrasecut_plus" else None,
        "unique_train_phrases": len(all_train_phrases), "queries_with_support": queries_with_support,
        "protocol": f"{TRAIN_MODE}; frozen CLIP; final epoch checkpoint; eval split held out"}
    write_json(OUT / "training_config.json", training_metadata)
    # Hình kiểm tra 4 mẫu PhraseCut+; fork_rng để không làm đổi RNG của lượt train.
    with torch.random.fork_rng(devices=[]):  # mẫu tạo trên CPU
        torch.manual_seed(SEED)
        preview = [PhraseCutPlusDataset()[i] for i in torch.randperm(len(train_tasks))[:4].tolist()]
    fig, axes = plt.subplots(len(preview), 3, figsize=(9, 3 * len(preview)), squeeze=False)
    for row, (image, prompt, support, valid, target) in zip(axes, preview):
        for ax, value, title in zip(row, [image, support, target], [f"Query: {prompt}", "Support" if valid else "Không có support", "Mask đáp án"]):
            shown = (value * image_std + image_mean).clamp(0, 1).permute(1, 2, 0) if value.ndim == 3 else value
            ax.imshow(shown, cmap="gray" if value.ndim == 2 else None, vmin=0, vmax=1)
            ax.set_title(title, fontsize=9)
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "phrasecut_plus_samples.png", dpi=100)
    plt.close(fig)
    print(f"Mode={TRAIN_MODE}; N_train={len(train_tasks)}; phrases={len(all_train_phrases)}; "
          f"queries with support from another image={queries_with_support}")
    print(f"N_train={len(train_tasks)}, epochs={EPOCHS}, batch={BATCH_SIZE}")
    print(f"steps/epoch={steps_per_epoch}, total optimizer steps={total_steps}")
else:
    print("TRAIN_DECODER=False: đánh giá checkpoint pretrained, không train.")


In [ ]:
LORA_LR_SCALE = 1.0
import math
from contextlib import contextmanager
import torch.utils.checkpoint

class LoRALinear(torch.nn.Module):
    """W·x + (alpha/r)·B·A·x; B khởi tạo 0 nên lúc đầu đúng bằng lớp gốc. Lớp gốc giữ đóng băng."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lora_a = torch.nn.Parameter(torch.empty(rank, base.in_features, device=base.weight.device))
        self.lora_b = torch.nn.Parameter(torch.zeros(base.out_features, rank, device=base.weight.device))
        torch.nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))
        self.scale = alpha / rank
        self.enabled = True

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + x @ self.lora_a.t().to(x.dtype) @ self.lora_b.t().to(x.dtype) * self.scale
        return out

def lora_layers(model):
    """Các layer CLIP vision có output đi vào decoder (0..max(extract_layers))."""
    return list(model.clip.vision_model.encoder.layers[:max(model.extract_layers) + 1])

def lora_modules(model):
    return [m for m in model.clip.vision_model.modules() if isinstance(m, LoRALinear)]

def has_lora(model):
    return bool(lora_modules(model))

def attach_lora(model, rank, alpha):
    assert not has_lora(model), 'LoRA đã được gắn'
    for layer in lora_layers(model):
        for name in ['q_proj', 'v_proj']:
            setattr(layer.self_attn, name, LoRALinear(getattr(layer.self_attn, name), rank, alpha))

def lora_parameters(model):
    return [p for m in lora_modules(model) for p in (m.lora_a, m.lora_b)]

def trainable_groups(model, lr):
    """Nhóm 0 = decoder (lr), nhóm 1 = LoRA (lr × LORA_LR_SCALE). Không có LoRA thì chỉ một nhóm, như B1."""
    groups = [{'params': list(model.decoder.parameters()), 'lr': lr}]
    if lora_parameters(model):
        groups.append({'params': lora_parameters(model), 'lr': lr * LORA_LR_SCALE})
    return groups

def sync_lora_lr(optimizer):
    """Giữ LR của LoRA đúng bằng LORA_LR_SCALE × LR decoder sau mỗi bước của lịch cosine."""
    if len(optimizer.param_groups) > 1:
        optimizer.param_groups[1]['lr'] = optimizer.param_groups[0]['lr'] * LORA_LR_SCALE

def lora_state_dict(model):
    return {k: v.detach().cpu() for k, v in model.clip.vision_model.state_dict().items() if 'lora_' in k}

def load_lora_state(model, state):
    missing, unexpected = model.clip.vision_model.load_state_dict(state, strict=False)
    assert not unexpected and (not [k for k in missing if 'lora_' in k]), (missing, unexpected)

@contextmanager
def lora_disabled(model):
    """Embedding điều kiện (text, ảnh support) luôn tính bằng CLIP gốc, kể cả ở E4c."""
    modules = lora_modules(model)
    for m in modules:
        m.enabled = False
    try:
        yield
    finally:
        for m in modules:
            m.enabled = True

def vision_activations(model, pixel_values):
    """Output các layer extract_layers của CLIP vision, chỉ chạy tới layer cuối cần dùng."""
    vision = model.clip.vision_model
    hidden = vision.pre_layrnorm(vision.embeddings(pixel_values, interpolate_pos_encoding=True))
    activations = []
    for index, layer in enumerate(lora_layers(model)):
        if LORA_GRAD_CHECKPOINT and torch.is_grad_enabled():
            hidden = torch.utils.checkpoint.checkpoint(layer, hidden, None, None, use_reentrant=False)[0]
        else:
            hidden = layer(hidden, None, None)[0]
        if index in model.extract_layers:
            activations.append(hidden)
    return activations

def mix_conditionals(text_cond, visual_cond, text_weights):
    """cond = w·text + (1−w)·ảnh (giống cell PhraseCut+; định nghĩa sớm để segment dùng được khi chỉ đánh giá)."""
    weights = text_weights.to(text_cond.device, text_cond.dtype)
    return text_cond * weights + visual_cond * (1 - weights)

def segment(model, pixel_values, input_ids, attention_mask, support_pixel_values=None, text_weights=None):
    """Logits [B,352,352] cho train lẫn inference, model gốc (B1) hoặc CLIP vision có LoRA (E4c)."""
    size = pixel_values.shape[0]
    lora = has_lora(model)
    with torch.no_grad():
        if not lora:
            hidden = model.clip.vision_model(pixel_values=pixel_values, output_hidden_states=True, interpolate_pos_encoding=True).hidden_states
            activations = [hidden[i + 1] for i in model.extract_layers]
        with lora_disabled(model):
            cond = model.get_conditional_embeddings(batch_size=size, input_ids=input_ids, attention_mask=attention_mask)
            if support_pixel_values is not None:
                visual = model.get_conditional_embeddings(batch_size=size, conditional_pixel_values=support_pixel_values)
                cond = mix_conditionals(cond, visual, text_weights)
    if lora:
        activations = vision_activations(model, pixel_values)
    logits = model.decoder(activations, cond).logits
    return logits.reshape(size, *logits.shape[-2:])

def experiment_logits(model, pixel_values, input_ids=None, attention_mask=None, conditional_pixel_values=None):
    """E2/E3 forward: text hoặc ảnh support; với LoRA, embedding điều kiện vẫn dùng CLIP gốc."""
    if not has_lora(model):
        return model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, conditional_pixel_values=conditional_pixel_values).logits
    if (input_ids is None) == (conditional_pixel_values is None):
        raise ValueError('Supply text or support images, not both')
    if input_ids is not None:
        return segment(model, pixel_values, input_ids, attention_mask)
    with torch.no_grad():
        activations = vision_activations(model, pixel_values)
        with lora_disabled(model):
            cond = model.get_conditional_embeddings(batch_size=pixel_values.shape[0], conditional_pixel_values=conditional_pixel_values)
    return model.decoder(activations, cond).logits


In [ ]:
if TRAIN_DECODER:
    if TRAIN_INIT == "random_decoder":
        model.decoder = type(model.decoder)(model.config)
        model.decoder.post_init()
        model.decoder = model.decoder.to(device)
    model.clip.requires_grad_(False)
    model.decoder.requires_grad_(True)
    optimizer = torch.optim.AdamW(model.decoder.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=MIN_LEARNING_RATE)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    signature = hashlib.sha256(json.dumps(training_metadata, sort_keys=True).encode()).hexdigest()
    start_epoch, global_step, history = 0, 0, []
    checkpoint_dir = OUT / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)
    if not RESUME_CHECKPOINT:
        for old_checkpoint in checkpoint_dir.glob("decoder_epoch_*.pt"):
            old_checkpoint.unlink()
    if RESUME_CHECKPOINT:
        checkpoint = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=True)
        if checkpoint["config_signature"] != signature:
            raise ValueError("Checkpoint khác train manifest/config; giữ nguyên cấu hình khi resume.")
        model.decoder.load_state_dict(checkpoint["decoder"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        start_epoch, global_step, history = checkpoint["epoch"], checkpoint["global_step"], checkpoint["history"]
        generator.set_state(checkpoint["loader_rng"].cpu())
        torch.set_rng_state(checkpoint["torch_rng"].cpu())
        random.setstate(checkpoint["python_rng"])
        state = checkpoint["numpy_rng"]
        np.random.set_state((state[0], state[1].numpy().astype(np.uint32), state[2], state[3], state[4]))
        if device.type == "cuda" and checkpoint["cuda_rng"] is not None:
            # Session resume có thể khác số GPU; chỉ khôi phục các GPU đang có.
            for index, value in enumerate(checkpoint["cuda_rng"][:torch.cuda.device_count()]):
                torch.cuda.set_rng_state(value.cpu(), index)

    class TrainForward(torch.nn.Module):
        """Một bước forward train. DataParallel chia batch của module này cho từng GPU."""
        def __init__(self, model):
            super().__init__()
            self.model = model

        def forward(self, pixel_values, input_ids, attention_mask, support_pixel_values=None, text_weights=None):
            size = pixel_values.shape[0]
            context = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
            with context:
                cond = self.model.get_conditional_embeddings(batch_size=size, input_ids=input_ids,
                                                             attention_mask=attention_mask)
                if support_pixel_values is not None:
                    visual_cond = self.model.get_conditional_embeddings(
                        batch_size=size, conditional_pixel_values=support_pixel_values)
                    cond = mix_conditionals(cond, visual_cond, text_weights)
                logits = self.model(pixel_values=pixel_values, conditional_embeddings=cond).logits
            # HF bỏ chiều batch khi batch = 1; giữ [B,H,W] để DataParallel gộp đúng.
            return logits.reshape(size, *logits.shape[-2:])

    train_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
    use_data_parallel = MULTI_GPU and train_gpus > 1
    train_forward = TrainForward(model)
    if use_data_parallel:
        train_forward = torch.nn.DataParallel(train_forward)
        assert BATCH_SIZE >= train_gpus, "BATCH_SIZE phải ≥ số GPU."
    print(f"Train trên {train_gpus if use_data_parallel else 1} {'GPU' if device.type == 'cuda' else 'CPU'}"
          f"{' (DataParallel, mỗi GPU ~' + str(math.ceil(BATCH_SIZE / train_gpus)) + ' mẫu/batch)' if use_data_parallel else ''}")
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        model.clip.eval()
        model.decoder.train()
        loss_sum, seen = 0.0, 0
        for index in range(train_gpus):
            torch.cuda.reset_peak_memory_stats(index)
        epoch_started = time.perf_counter()
        for batch in tqdm(train_loader, desc=f"Train epoch {epoch + 1}/{EPOCHS}"):
            optimizer.zero_grad(set_to_none=True)
            inputs = {key: batch[key] for key in ["pixel_values", "input_ids", "attention_mask"]}
            if TRAIN_MODE == "phrasecut_plus":
                inputs["support_pixel_values"] = batch["support_pixel_values"]
                inputs["text_weights"] = sample_text_weights(batch["support_valid"], MIX_TEXT_MAX)
            if not use_data_parallel:
                # DataParallel tự chép từng phần batch từ CPU sang đúng GPU; một GPU thì chép trước.
                inputs = {key: value.to(device, non_blocking=True) for key, value in inputs.items()}
            logits = train_forward(**inputs)
            loss = F.binary_cross_entropy_with_logits(logits.float(), batch["labels"].to(logits.device, non_blocking=True))
            if not torch.isfinite(loss):
                raise ValueError("Training loss NaN/Inf; dừng để kiểm tra dữ liệu/AMP.")
            old_scale = scaler.get_scale()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # AMP có thể bỏ optimizer step khi overflow; không tăng lịch LR lúc đó.
            if not USE_AMP or scaler.get_scale() >= old_scale:
                scheduler.step()
                global_step += 1
            count = batch["labels"].shape[0]
            loss_sum += loss.detach().item() * count
            seen += count
        assert seen == len(train_tasks), "Epoch chưa duyệt đủ train queries."
        synchronize()
        history.append({"epoch": epoch + 1, "train_loss": loss_sum / seen, "n_queries": seen,
            "optimizer_steps_total": global_step, "learning_rate_next": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.perf_counter() - epoch_started, "gpus": train_gpus if use_data_parallel else min(train_gpus, 1),
            "peak_gpu_mb": [round(torch.cuda.max_memory_allocated(i) / 1024**2) for i in range(train_gpus)]})
        np_state = np.random.get_state()
        checkpoint = {"epoch": epoch + 1, "global_step": global_step, "config_signature": signature,
            "decoder": model.decoder.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "history": history,
            "loader_rng": generator.get_state(), "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
            "python_rng": random.getstate(),
            "numpy_rng": (np_state[0], torch.from_numpy(np_state[1].astype(np.int64)), np_state[2], np_state[3], np_state[4]),
            "base_model_id": MODEL_ID, "base_model_revision": MODEL_REVISION,
            "training": training_metadata}  # E2/E3 đọc epoch, batch size từ đây
        checkpoint_path = checkpoint_dir / f"decoder_epoch_{epoch + 1:03d}.pt"
        torch.save(checkpoint, checkpoint_path)
        write_csv(OUT / "training_history.csv", history)
        print(history[-1])
    assert start_epoch <= EPOCHS, "Checkpoint vượt EPOCHS được yêu cầu."
    model.eval()
    training_metadata.update({"completed_epochs": EPOCHS, "optimizer_steps_completed": global_step,
        "final_checkpoint": str(RESUME_CHECKPOINT) if start_epoch == EPOCHS else str(checkpoint_path)})
    write_json(OUT / "training_config.json", training_metadata)
    # Backbone không nhận gradient; chỉ decoder có tham số trainable.
    assert all(p.grad is None for p in model.clip.parameters())
    print(f"Training complete. Evaluate final epoch on {EVAL_SPLIT} next.")
elif DECODER_CHECKPOINT:
    # Chỉ đánh giá: thay decoder tác giả bằng decoder đã train trong một lượt E1 trước.
    checkpoint = torch.load(DECODER_CHECKPOINT, map_location="cpu", weights_only=True)
    if checkpoint.get("base_model_id") != MODEL_ID or checkpoint.get("base_model_revision") != MODEL_REVISION:
        raise ValueError("Checkpoint được train trên base model/revision khác MODEL_ID/MODEL_REVISION.")
    if checkpoint.get("lora"):
        adapter = (checkpoint.get("training") or {}).get("lora") or {}
        attach_lora(model, adapter.get("rank", LORA_RANK), adapter.get("alpha", LORA_ALPHA))
        load_lora_state(model, checkpoint["lora"])
    model.decoder.load_state_dict(checkpoint["decoder"])
    model.eval()
    training_metadata = {"enabled": False, "decoder_checkpoint": str(DECODER_CHECKPOINT),
        "decoder_checkpoint_sha256": sha256(Path(DECODER_CHECKPOINT)), "decoder_epoch": checkpoint["epoch"],
        "decoder_training": checkpoint.get("training"), "decoder_history": checkpoint.get("history")}
    del checkpoint
    print(f"Đánh giá decoder epoch {training_metadata['decoder_epoch']} từ {DECODER_CHECKPOINT}")
else:
    print("Đánh giá decoder pretrained của tác giả (CIDAS/clipseg-rd64-refined).")


In [ ]:
if has_lora(model):
    # Forward của Transformers (dùng trong predict_batch) phải trùng segment của lúc train/chấm E4c.
    with torch.no_grad():
        probe = processor.tokenizer(["a red car", "dog"], padding=True, return_tensors="pt").to(device)
        probe_pixels = torch.randn(2, 3, PAPER_SIZE, PAPER_SIZE, device=device)
        assert torch.allclose(model(pixel_values=probe_pixels, **probe).logits,
                              segment(model, probe_pixels, probe["input_ids"], probe["attention_mask"]), atol=1e-4)
    print(f"LoRA: {len(lora_modules(model))} lớp, {sum(p.numel() for p in lora_parameters(model)):,} tham số; "
          "forward khớp segment")
else:
    print("Decoder không có LoRA")


## E5. SAM và các hàm tinh chỉnh

Prompt tính từ xác suất CLIPSeg ở độ phân giải gốc. Toạ độ được đổi sang khung 1024 của SAM
(cạnh dài = 1024, như `SamProcessor`). Logit SAM được đưa về kích thước gốc bằng `post_process_masks`,
gộp các vùng bằng max, rồi resize bilinear về 352×352 để chấm theo protocol paper.


In [ ]:
from scipy import ndimage
from transformers import SamModel, SamProcessor

sam_processor = SamProcessor.from_pretrained(SAM_MODEL)
sam_model = SamModel.from_pretrained(SAM_MODEL).to(device).eval()
sam_commit = getattr(sam_model.config, "_commit_hash", None)
print("SAM:", SAM_MODEL, sam_commit, "| tham số:", sum(p.numel() for p in sam_model.parameters()))
PAPER_T03 = int(np.argmin(np.abs(PAPER_THRESHOLDS - THRESHOLD)))

def prompt_components(probability, threshold=THRESHOLD):
    """Box [x0, y0, x1, y1] và điểm [x, y] (tâm pixel có xác suất cao nhất) cho từng vùng liên thông p >= threshold."""
    labels, count = ndimage.label(probability >= threshold)
    if count == 0:
        return []
    areas = np.bincount(labels.ravel())[1:]
    order = np.argsort(-areas, kind="stable")
    prompts = []
    for index in order[:SAM_MAX_COMPONENTS]:
        if areas[index] < SAM_MIN_COMPONENT_RATIO * areas[order[0]]:
            break
        region = labels == index + 1
        ys, xs = np.nonzero(region)
        y, x = np.unravel_index(np.where(region, probability, -1.0).argmax(), probability.shape)
        prompts.append(([float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)],
                        [float(x) + 0.5, float(y) + 0.5]))
    return prompts

def sam_coordinate_scale(height, width, longest_edge=1024):
    """Hệ số đổi toạ độ (x, y) sang khung SAM, làm tròn kích thước như SamProcessor._normalize_coordinates."""
    scale = longest_edge / max(height, width)
    new_height, new_width = int(height * scale + 0.5), int(width * scale + 0.5)
    return new_width / width, new_height / height

@torch.inference_mode()
def sam_image(image):
    """Embedding SAM một lần cho mỗi ảnh, kèm kích thước để đổi toạ độ và hậu xử lý mask."""
    inputs = sam_processor(images=image, return_tensors="pt")
    embeddings = sam_model.get_image_embeddings(inputs["pixel_values"].to(device))
    return {"embeddings": embeddings, "original_sizes": inputs["original_sizes"],
            "reshaped_input_sizes": inputs["reshaped_input_sizes"]}

@torch.inference_mode()
def sam_logits(sam_state, prompts, use_points):
    """Logit SAM ở kích thước gốc [H, W]: mỗi vùng một mask (multimask_output=False), gộp bằng max."""
    height, width = (int(v) for v in sam_state["original_sizes"][0])
    sx, sy = sam_coordinate_scale(height, width)
    boxes = torch.tensor([[box for box, _ in prompts]], dtype=torch.float32) * torch.tensor([sx, sy, sx, sy])
    kwargs = {"input_boxes": boxes.to(device)}
    if use_points:
        points = torch.tensor([[[point] for _, point in prompts]], dtype=torch.float32) * torch.tensor([sx, sy])
        kwargs["input_points"] = points.to(device)
        kwargs["input_labels"] = torch.ones((1, len(prompts), 1), dtype=torch.long, device=device)
    outputs = sam_model(image_embeddings=sam_state["embeddings"], multimask_output=False, **kwargs)
    masks = sam_processor.image_processor.post_process_masks(outputs.pred_masks.cpu(), sam_state["original_sizes"],
        sam_state["reshaped_input_sizes"], binarize=False)[0]          # [n_prompts, 1, H, W] logits
    return masks[:, 0].amax(0).float()

def to_paper_size(logits):
    """Logit [H, W] ở kích thước gốc → xác suất 352×352 (bilinear trên logit rồi sigmoid)."""
    resized = F.interpolate(logits[None, None], size=(PAPER_SIZE, PAPER_SIZE), mode="bilinear", align_corners=False)
    return resized[0, 0].sigmoid().numpy()

def variant_probabilities(clip_352, sam_352_box, sam_352_point):
    """Năm biến thể ở 352×352. Không có prompt (sam_* = None) thì giữ nguyên CLIPSeg."""
    box = clip_352 if sam_352_box is None else sam_352_box
    point = clip_352 if sam_352_point is None else sam_352_point
    return {"clipseg": clip_352, "sam_box_replace": box, "sam_box_mean": 0.5 * (clip_352 + box),
            "sam_boxpoint_replace": point, "sam_boxpoint_mean": 0.5 * (clip_352 + point)}

def query_iou(stats):
    tp, fp, fn = stats[0, PAPER_T03], stats[1, PAPER_T03], stats[2, PAPER_T03]
    return float(tp / (1 + tp + fp + fn))

def paired_bootstrap(base, other, image_of, reps=5000, seed=SEED):
    """Δ mIoU@0,3 (other − base) ghép theo query, CI 95% bootstrap theo ảnh."""
    ids = sorted(base)
    delta = np.array([other[q] - base[q] for q in ids])
    images = sorted({image_of[q] for q in ids})
    position = {image: i for i, image in enumerate(images)}
    groups = np.array([position[image_of[q]] for q in ids])
    sums, counts = np.bincount(groups, delta, len(images)), np.bincount(groups, minlength=len(images))
    draws = np.random.default_rng(seed).integers(0, len(images), size=(reps, len(images)))
    boot = sums[draws].sum(1) / counts[draws].sum(1)
    return {"delta_miou_0.3": float(delta.mean()), "ci95": [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
            "wins": int((delta > 1e-9).sum()), "losses": int((delta < -1e-9).sum()), "n_queries": len(ids)}

def run_split():
    """Chấm mọi biến thể trên split hiện tại (EVAL_SPLIT); lưu stats, metric từng query và summary."""
    split_dir = OUT / "splits" / EVAL_SPLIT
    split_dir.mkdir(parents=True, exist_ok=True)
    tasks_by_image = defaultdict(list)
    for task in selected:
        tasks_by_image[int(task["image_id"])].append(task)
    stats = {v: [] for v in VARIANTS}
    rows, fallback, started = [], 0, time.perf_counter()
    for image_id in tqdm(image_ids, desc=f"CLIPSeg+SAM {EVAL_SPLIT}"):
        with Image.open(image_paths[image_id]) as raw:
            image = raw.convert("RGB")
        state = sam_image(image)
        current = tasks_by_image[image_id]
        for start in range(0, len(current), BATCH_SIZE):
            batch = current[start:start + BATCH_SIZE]
            probabilities, probabilities_352, _ = predict_batch(image, batch, with_paper=True)
            for task, probability, clip_352 in zip(batch, probabilities, probabilities_352):
                prompts = prompt_components(probability)
                if prompts:
                    sam_box = to_paper_size(sam_logits(state, prompts, use_points=False))
                    sam_point = to_paper_size(sam_logits(state, prompts, use_points=True))
                else:
                    sam_box = sam_point = None
                    fallback += 1
                target = paper_target(rasterize_gt(task, image.width, image.height))
                row = {"task_id": str(task["task_id"]), "image_id": image_id, "n_prompts": len(prompts)}
                for variant, probability_352 in variant_probabilities(clip_352, sam_box, sam_point).items():
                    query_stats = paper_threshold_stats(probability_352, target)
                    stats[variant].append(query_stats)
                    row[f"iou_0.3_{variant}"] = query_iou(query_stats)
                rows.append(row)
        if len(rows) % 2000 < len(current):
            print(f"{EVAL_SPLIT}: {len(rows)}/{len(selected)} query, {time.perf_counter() - started:.0f}s", flush=True)
    assert len(rows) == len(selected), "Chưa chấm đủ query."
    write_csv(split_dir / "sample_metrics.csv", rows)
    np.savez_compressed(split_dir / "paper_threshold_stats.npz", thresholds=PAPER_THRESHOLDS,
                        task_ids=np.array([r["task_id"] for r in rows]),
                        **{variant: np.stack(values) for variant, values in stats.items()})
    image_of = {r["task_id"]: r["image_id"] for r in rows}
    iou = {v: {r["task_id"]: r[f"iou_0.3_{v}"] for r in rows} for v in VARIANTS}
    result = {"split": EVAL_SPLIT, "n_queries": len(rows), "complete_official_split": len(selected) == len(tasks),
              "fallback_no_prompt": fallback, "seconds": time.perf_counter() - started,
              "paper_protocol": {v: paper_metrics(np.stack(stats[v])) for v in VARIANTS},
              "paired_vs_clipseg": {v: paired_bootstrap(iou["clipseg"], iou[v], image_of) for v in VARIANTS[1:]}}
    write_json(split_dir / "summary.json", result)
    for v in VARIANTS:
        m = result["paper_protocol"][v]
        extra = "" if v == "clipseg" else (f" | Δ {100 * result['paired_vs_clipseg'][v]['delta_miou_0.3']:+.2f} "
            f"[{100 * result['paired_vs_clipseg'][v]['ci95'][0]:+.2f}, {100 * result['paired_vs_clipseg'][v]['ci95'][1]:+.2f}]")
        print(f"{EVAL_SPLIT} {v:22s} mIoU {100 * m['miou_0.3']:.2f} | IoU_FG {100 * m['fgiou_0.3']:.2f} | AP {100 * m['ap']:.2f}{extra}")
    print(f"Query không có vùng p ≥ {THRESHOLD} (giữ CLIPSeg): {fallback}")
    return result


## E5. Chấm val và chốt biến thể (trước khi chấm test)


In [ ]:
val_result = run_split()
# Quy tắc chọn, cố định trước khi chấm test: biến thể SAM có mIoU@0,3 trên val cao nhất.
candidates = VARIANTS[1:]
SELECTED_VARIANT = max(candidates, key=lambda v: (val_result["paper_protocol"][v]["miou_0.3"], -candidates.index(v)))
selection = {"rule": "highest val paper mIoU@0.3 among SAM variants; ties -> earlier in VARIANTS",
             "selected_variant": SELECTED_VARIANT, "val_paired_vs_clipseg": val_result["paired_vs_clipseg"][SELECTED_VARIANT],
             "improves_on_val": val_result["paired_vs_clipseg"][SELECTED_VARIANT]["ci95"][0] > 0}
write_json(OUT / "selection.json", selection)
print("Chốt trên val:", json.dumps(selection, ensure_ascii=False))


## E5. Chấm test bằng biến thể đã chốt (kèm mọi biến thể để minh bạch)


In [ ]:
EVAL_SPLIT = "test"
print("Chấm test: mọi biến thể, biến thể chính đã chốt trên val =", SELECTED_VARIANT)


In [ ]:
# Drive ID của annotation chính thức, theo download_dataset.py của PhraseCut.
REFER_DRIVE_IDS = {"miniv": "1oLcTQ1blTIQyu5ZMelQN2HSniuQaxM4E", "val": "1UyojArOFPlsSeNbA9fHWjCjOOU-OCohG",
                   "test": "1jrzXm1gcq6f5hNDeamZd0UmyyHUv61IZ", "train": "1qx-0q6r9r0YUGpoyT0B8HJKmUFWQDSu7"}

def metadata_path(root):
    for name in ["image_data_split.json", "image_data_split3000.json"]:
        candidate = root / name
        if candidate.is_file():
            return candidate
    return root / "image_data_split.json"

if DATA_ROOT is None:
    candidates = sorted(Path("/kaggle/input").rglob(f"refer_{EVAL_SPLIT}.json")) if Path("/kaggle/input").exists() else []
    candidates = [p.parent for p in candidates if metadata_path(p.parent).is_file()]
    if len(candidates) > 1:
        raise ValueError(f"Có nhiều dataset chứa refer_{EVAL_SPLIT}.json; đặt DATA_ROOT rõ ràng: " + str(candidates))
    data_root = candidates[0] if candidates else WORK_ROOT / "phrasecut_annotations"
else:
    data_root = Path(DATA_ROOT)
    if not data_root.is_dir():
        raise FileNotFoundError(f"DATA_ROOT không tồn tại: {data_root}")
data_root.mkdir(parents=True, exist_ok=True)
annotation_path = data_root / f"refer_{EVAL_SPLIT}.json"
info_path = metadata_path(data_root)

def ensure_json(path, file_id):
    if path.is_file():
        value = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError(f"JSON không phải danh sách annotation: {path}")
        return value
    if str(path).startswith("/kaggle/input/"):
        raise FileNotFoundError(f"Thiếu {path.name} trong input read-only; bổ sung Kaggle Dataset.")
    temp = path.with_suffix(".download")
    try:
        result = gdown.download(id=file_id, output=str(temp), quiet=False)
        if result is None:
            raise RuntimeError("Google Drive không trả về file.")
        value = json.loads(temp.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError("Nội dung download không phải annotation JSON.")
        temp.replace(path)
        return value
    except Exception as exc:
        raise RuntimeError(f"Không tải được {path.name}. Bật Internet, hoặc Add Input dataset và đặt DATA_ROOT.") from exc

tasks = ensure_json(annotation_path, REFER_DRIVE_IDS[EVAL_SPLIT])
image_info = ensure_json(info_path, "1xB9eExJo35K3DQu8PnBWZ1q_OMoMl8-i")
info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == EVAL_SPLIT}
assert len({str(t["task_id"]) for t in tasks}) == len(tasks), "Task ID bị trùng."
assert all(int(t["image_id"]) in info_by_id for t in tasks), f"Annotation không thuộc split {EVAL_SPLIT}."
assert all(t.get("Polygons") and t.get("phrase", "").strip() for t in tasks), "Thiếu polygon GT hoặc phrase."
assert all(re.fullmatch(r"[A-Za-z0-9_-]+", str(t["task_id"])) for t in tasks), "Task ID không hợp lệ."

selected = sorted(tasks, key=lambda t: str(t["task_id"]))
if MAX_QUERIES is not None and MAX_QUERIES < len(selected):
    selected = random.Random(SEED).sample(selected, MAX_QUERIES)
selected.sort(key=lambda t: (int(t["image_id"]), str(t["task_id"])))
image_ids = sorted({int(t["image_id"]) for t in selected})
if len(image_ids) < 2:
    raise ValueError("Cần ít nhất 2 image_id để chia calibration/evaluation; tăng MAX_QUERIES.")
shuffled_ids = image_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)
n_cal = min(len(image_ids) - 1, max(1, round(len(image_ids) * CALIBRATION_FRACTION)))
calibration_ids = set(shuffled_ids[:n_cal])
group_for_image = {i: "calibration" if i in calibration_ids else "evaluation" for i in image_ids}
assert not calibration_ids.intersection(set(image_ids) - calibration_ids)
write_json(OUT / "selected_tasks.json", [{"task_id": str(t["task_id"]), "image_id": int(t["image_id"]),
    "phrase": t["phrase"], "group": group_for_image[int(t["image_id"])]} for t in selected])
print(f"Annotation {EVAL_SPLIT}: {len(tasks)} queries / {len(info_by_id)} images")
print(f"Selected: {len(selected)} queries / {len(image_ids)} images; calibration {n_cal} images")


In [ ]:
image_cache = WORK_ROOT / "phrasecut_image_cache"
image_cache.mkdir(parents=True, exist_ok=True)
def image_search_roots(*extra):
    """Nơi tìm ảnh <image_id>.jpg: IMAGE_ROOT nếu đặt, nếu không thì mọi Kaggle Input; cộng thư mục annotation."""
    roots = [Path(IMAGE_ROOT)] if IMAGE_ROOT else ([Path("/kaggle/input")] if Path("/kaggle/input").exists() else [])
    roots += [Path(r) for r in extra]
    # Bỏ thư mục nằm trong thư mục đã quét để không quét hai lần.
    return [r for i, r in enumerate(roots) if r.is_dir() and not any(r == q or q in r.parents for q in roots[:i])]

def index_local_images(roots, wanted_ids):
    index = {}
    for search_root in roots:
        for path in sorted(search_root.rglob("*")):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and path.stem.isdigit() and int(path.stem) in wanted_ids:
                index.setdefault(int(path.stem), path)
    return index

local_images = index_local_images(image_search_roots(data_root), set(image_ids))
print(f"Ảnh có sẵn trong Input: {len(local_images)}/{len(image_ids)}; phần còn lại tải từ Visual Genome")

def valid_image(path, info):
    with Image.open(path) as image:
        image.load()
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"{path.name}: ảnh {image.size} != metadata {(info['width'], info['height'])}")

def prepare_image(image_id):
    info = info_by_id[image_id]
    path = local_images.get(image_id, image_cache / f"{image_id}.jpg")
    if path.is_file():
        valid_image(path, info)
        return image_id, path
    url = info["url"].replace("http://", "https://", 1)
    temp = path.with_suffix(".download")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with temp.open("wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        f.write(chunk)
            valid_image(temp, info)
            temp.replace(path)
            return image_id, path
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

image_paths, image_errors = {}, []
def prepare_safe(image_id):
    try:
        return prepare_image(image_id), None
    except Exception as exc:
        return None, {"image_id": image_id, "error": str(exc)}

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    for result, error in tqdm(pool.map(prepare_safe, image_ids), total=len(image_ids), desc="Images"):
        if error:
            image_errors.append(error)
        else:
            image_paths[result[0]] = result[1]
write_json(OUT / "image_errors.json", image_errors)
if image_errors:
    raise RuntimeError(f"{len(image_errors)} ảnh lỗi; xem image_errors.json. Sửa dữ liệu rồi chạy lại.")
print("Validated images:", len(image_paths))


In [ ]:
test_result = run_split()
clipseg_test = test_result["paper_protocol"]["clipseg"]
reproduces = (all(abs(clipseg_test[k] - v) <= 0.0002 for k, v in CLIPSEG_REFERENCE_TEST.items())
              if MAX_QUERIES is None else None)
final = {"decoder_label": DECODER_LABEL, "decoder_checkpoint_sha256": DECODER_SHA256, "has_lora": has_lora(model),
         "sam_model": SAM_MODEL, "sam_commit": sam_commit,
         "settings": {"threshold": THRESHOLD, "max_components": SAM_MAX_COMPONENTS,
                      "min_component_ratio": SAM_MIN_COMPONENT_RATIO, "fusions": ["replace", "mean"]},
         "selection": selection, "val": val_result, "test": test_result,
         "clipseg_test_reproduces_reference": reproduces, "clipseg_reference_test": CLIPSEG_REFERENCE_TEST,
         "clipseg_test_reproduces_e1": reproduces if DECODER_LABEL == "e1" else None}
write_json(OUT / "summary.json", final)
lines = ["| Biến thể | val mIoU | val Δ (95% CI) | test mIoU | test IoU_FG | test AP | test Δ (95% CI) |",
         "|---|---:|---:|---:|---:|---:|---:|"]
for v in VARIANTS:
    val_m, test_m = val_result["paper_protocol"][v], test_result["paper_protocol"][v]
    def delta(result):
        if v == "clipseg":
            return "—"
        d = result["paired_vs_clipseg"][v]
        return f"{100 * d['delta_miou_0.3']:+.2f} [{100 * d['ci95'][0]:+.2f}, {100 * d['ci95'][1]:+.2f}]"
    mark = " (chốt)" if v == SELECTED_VARIANT else ""
    lines.append(f"| {v}{mark} | {100 * val_m['miou_0.3']:.2f} | {delta(val_result)} | {100 * test_m['miou_0.3']:.2f} | "
                 f"{100 * test_m['fgiou_0.3']:.2f} | {100 * test_m['ap']:.2f} | {delta(test_result)} |")
(OUT / "e5_comparison.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
print(f"Dòng clipseg trên test trùng số đã biết của {DECODER_LABEL}:", reproduces)
shutil.copy2(OUT / "summary.json", OUT / "splits" / "summary_all.json")


In [ ]:
archive = WORK_ROOT / "clipseg_e5_sam.zip"
archive.unlink(missing_ok=True)
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and path.suffix != ".png":
            package.write(path, arcname=path.relative_to(OUT).as_posix())
print("Download:", archive)
